# Models Annotations

`this Jupyter notebook provides all my decision making behind the models.py file for the backend of ApolloTune`

### USERPROFILE TABLE

In [ ]:
from django.db import models
from django.contrib.auth.models import User
class UserProfile(models.Model):
    """
    This class extends Djangos built-in User model to  add Tuner/Creator distinction
    This follows a one-to-one relationship with the User model.
    """
    USER_TYPE_CHOICES = [
        ('tuner', 'Tuner'), #Listeners
        ('creator', 'Creator'), #Broadcasters
    ]

    user = models.OneToOneField(User, on_delete=models.CASCADE, related_name='profile')
    user_type = models.CharField(max_length=10, choices=USER_TYPE_CHOICES)
    bio = models.TextField(blank=True, null=True)
    created_at = models.DateTimeField(auto_now_add=True)


    def __str__(self):
        return f"{self.user.username} - {self.user_type}"

`This Code Defines the UserProfile Table, it extends Djangos UserModel to allow for the use of both TUNERS and CREATORS`<br>
`The Table can have two different types, either a TUNER or a CREATOR, tuners listen in onto the broadcast, Creators can start broadcasts`


In [ ]:
class Channel(models.Model):
    """
    Each Creator can have their own induvidual Channel.
    """
    creator = models.OneToOneField(
        User, 
        on_delete=models.CASCADE, 
        related_name='channel', 
        limit_choices_to={'profile__user_type': 'creator'}
    )
    name = models.CharField(max_length=100)
    description = models.TextField(blank=True, null=True)
    genre = models.CharField(max_length=50, blank=True, null=True)
    is_live = models.BooleanField(default=False)
    created_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"{self.name} by {self.creator.user.username}"



`Channel defines the Creators Channel, think of it as a youtube channel, like DanTDM, has a channel associated to his account (UserProfile)`<br>
`We state that the creator table is a 1-to-1 relationship with Django's built in User model, the on_delete means that when we delete a user, their channel is deleted.`


In [ ]:
class Broadcast(models.Model):
    """
    Individual Broadcast sessions for each Channel.
    Tracks when a creator goes live and the details of that session.
    """
    channel = models.ForeignKey(Channel, on_delete=models.CASCADE, related_name='broadcasts')
    title = models.CharField(max_length=200)
    description = models.TextField(blank=True, null=True)
    start_time = models.DateTimeField(auto_now_add=True)
    end_time = models.DateTimeField(blank=True, null=True)
    is_active = models.BooleanField(default=True)
    current_listeners = models.PositiveIntegerField(default=0)



`having a table to store this information allows us to retrieve information about a Channels previous broadcasts`<br>
`This Helps us detect which broadcast the user is connected to.`


In [ ]:
class ChatMessage(models.Model):
    """
    Chat Messages sent by Tuners during a broadcast
    """
    broadcast = models.ForeignKey(Broadcast, on_delete=models.CASCADE, related_name='messages')
    user = models.ForeignKey(User, on_delete=models.CASCADE)
    message = models.TextField()
    timestamp = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ['timestamp']

    def __str__(self):
        return f"Message by {self.user.username} at {self.timestamp}"

`Here, each chat message is linked through what Broadcast it was sent to, hence thats why broadcast is the foreign key in this table` <br>
`The class meta defined extra options/behaviour for your model that arent fields. It tells Django how to handle the model`<br>
`When we call the ChatMessage.objects.all(), it will already be ordered based on timestamp, for oldest to newest, when we want to display it on the screen.`



In [ ]:
class Listener(models.Model):
    """
    Tracks which Tuners are listening to which Broadcasts.
    """
    broadcast = models.ForeignKey(Broadcast, on_delete=models.CASCADE, related_name='listeners')
    user = models.ForeignKey(User, on_delete=models.CASCADE)
    joined_at = models.DateTimeField(auto_now_add=True)
    left_at = models.DateTimeField(blank=True, null=True)
   
    class Meta:
        unique_together = ('broadcast', 'user')

    def __str__(self):
        return f"{self.user.username} listening to {self.broadcast.title}"

`this is the class for the listener, it tracks what Tuners are listening to what broadcast. it has two foreign keys forming  acomposite key.`
`Here we use the meta class again, Where we bind broadcast and user meaning that only a TUNER can only LISTEN to 1 BROADCAST at a TIME`